In [22]:
"""
Q3.1  网格搜索：每台过滤器最小 EAC 的 (T_M, T_L)

EAC_i(T_M, T_L) = (c_buy + n_M * c_m + n_L * c_l) / L_years

其中:
  c_buy = 300 万元 (新购)
  c_m   =   3 万元 (中维护)
  c_l   =  12 万元 (大维护)
  L_years = 寿命（从 2026-01-20 起到滚动 365 日均值首次 < 37）
  n_M   = L_years · 365.25 / T_M (整个寿命内中维护次数)
  n_L   = L_years · 365.25 / T_L (若 T_L 有限，否则 n_L = 0)

策略:
  为避免长寿命过滤器无法给出 L，仿真地平线延长到 20 年 (H=7300)。
  若 20 年内仍未退役，记 L = 20（上限），EAC 为保守估计（真实 EAC 更低）。

网格:
  T_M ∈ {30, 40, 50, 60, 70, 80, 90, 100, 120} (9)
  T_L ∈ {120, 180, 240, 300, 360, 450, 540, 720, ∞} (9)
  共 81 个组合 × 10 台 = 810 次模拟。
"""
import pandas as pd
import numpy as np
from pathlib import Path
import time

PROJECT_ROOT = Path(r"d:\Users\32174\Desktop\2026_math_modeling_competition")
ROOT = PROJECT_ROOT  # 修正：指向项目根目录

# -------- 载入 Q2 模型系数 --------
coef = pd.read_csv(ROOT / "Q2输出/tables/Q2_winner_coeffs.csv")
beta_dict = dict(zip(coef["var"], coef["beta"]))
print(f"Loaded {len(beta_dict)} coefficients from Q2")

# -------- 历史数据 --------
mnt = pd.read_csv(ROOT / "Q1输出/data/maintenance_events.csv", parse_dates=["d"])
hist = pd.read_csv(ROOT / "Q1输出/data/daily_with_vars.csv", parse_dates=["d"])


Loaded 28 coefficients from Q2


In [23]:

# ===== 关键：时间起点必须与 Q2 模型训练时一致 =====
# Q2 模型使用 hist["t"] 列，其 t=0 对应 hist["d"].min()
anchor = hist["d"].min()  # 2024-04-03
print(f"Anchor (t=0): {anchor}")

# -------- 仿真参数 --------
start_future = pd.Timestamp("2026-01-20")
# 注意：题目没有要求 20 年地平线，仿真到自然退役为止
# 为避免无限循环，设置一个合理的最大仿真天数（例如 50 年 = 18250 天）
H_MAX = 18250  # 50 年上限
future_days = pd.date_range(start_future, periods=H_MAX, freq="D")
n_future = len(future_days)
T_year = 365.25
future_t = (future_days - anchor).days.values.astype(float)
future_sin1 = np.sin(2*np.pi*future_t/T_year)
future_cos1 = np.cos(2*np.pi*future_t/T_year)
d_arr = future_days.values  # numpy datetime64

# -------- 成本常数 --------
C_BUY = 300.0
C_M = 3.0
C_L = 12.0

# -------- 网格 --------
T_M_grid = [30, 40, 50, 60, 70, 80, 90, 100, 120]
T_L_grid = [120, 180, 240, 300, 360, 450, 540, 720, np.inf]


Anchor (t=0): 2024-04-03 00:00:00


In [24]:

# -------- 预合并观测 y （用于滚动 365 日均值的前置窗口） --------
hist_slim = hist[["i","d","y"]].copy().rename(columns={"y":"y_obs"})

# -------- 构建 H_window 辅助 --------
def H_window_from_dates(dates, w=7):
    """给定维护日期列表，返回 n_future 长的 0/1 数组"""
    flag = np.zeros(n_future, dtype=np.int8)
    for tau in dates:
        tau64 = np.datetime64(tau)
        tau_end = np.datetime64(tau + pd.Timedelta(days=w))
        mask = (d_arr > tau64) & (d_arr <= tau_end)
        flag[mask] = 1
    return flag

def A_from_dates(dates):
    """给定维护日期列表（含历史+未来，且 <= future_days[-1]），返回每日 days-since-last"""
    out = np.full(n_future, np.nan)
    dates_sorted = sorted(dates)
    if not dates_sorted:
        return out
    # 对每个 future day，找最后一个 <= d 的 dates
    # 利用 searchsorted
    dates_np = np.array([np.datetime64(x) for x in dates_sorted])
    idx = np.searchsorted(dates_np, d_arr, side="right") - 1
    valid = idx >= 0
    if valid.any():
        last = dates_np[idx[valid]]
        diff = (d_arr[valid] - last).astype("timedelta64[D]").astype(float)
        out[valid] = diff
    return out

def simulate_y(i, T_M, T_L):
    """给定 (i, T_M, T_L)，返回 n_future 长 y_sim 数组"""
    past = mnt[mnt["i"] == i].sort_values("d")
    past_m = past[past["q"] == "m"]["d"].tolist()
    past_l = past[past["q"] == "l"]["d"].tolist()

    last_m = past_m[-1] if past_m else anchor
    last_l = past_l[-1] if past_l else None

    future_m = []
    nxt = last_m + pd.Timedelta(days=int(round(T_M)))
    while nxt <= future_days[-1]:
        future_m.append(nxt)
        nxt += pd.Timedelta(days=int(round(T_M)))

    future_l = []
    if np.isfinite(T_L):
        base_l = last_l if last_l is not None else last_m  # 若无大维护，从上次中维护起步
        nxt = base_l + pd.Timedelta(days=int(round(T_L)))
        while nxt <= future_days[-1]:
            future_l.append(nxt)
            nxt += pd.Timedelta(days=int(round(T_L)))

    all_m = past_m + future_m
    all_l = past_l + future_l

    u_m = np.isin(d_arr, np.array([np.datetime64(x) for x in future_m])).astype(np.int8)
    u_l = np.isin(d_arr, np.array([np.datetime64(x) for x in future_l])).astype(np.int8) if future_l else np.zeros(n_future, dtype=np.int8)

    H_m7 = H_window_from_dates(all_m, w=7)
    H_l7 = H_window_from_dates(all_l, w=7) if all_l else np.zeros(n_future, dtype=np.int8)

    A_m = A_from_dates(all_m)
    A_l = A_from_dates(all_l) if all_l else np.full(n_future, np.nan)
    has_m = (~np.isnan(A_m)).astype(np.int8)
    has_l = (~np.isnan(A_l)).astype(np.int8)
    A_m_f = np.where(np.isnan(A_m), 0.0, A_m)
    A_l_f = np.where(np.isnan(A_l), 0.0, A_l)

    # === 应用论文 Eq.(1) 的 28 个回归系数 (β 来自 Q2 winner_coeffs.csv) ===
    # 论文符号 ↔ 代码:
    #   α_i  →  const + I(i={i})   (filter intercept)
    #   β_i·t →  t_i{i} * t         (filter time slope)
    #   γ_1, γ_2 →  sin1, cos1     (季节)
    #   δ_m, δ_l →  H_m7, H_l7     (维护脉冲)
    #   ρ_m, ρ_l →  A_m_f, A_l_f   (距离衰减)
    #   η_m, η_l →  has_m, has_l   (启动哑变量)
    y = np.full(n_future, beta_dict.get("const", 0.0))
    if f"I(i={i})" in beta_dict:
        y += beta_dict[f"I(i={i})"]                # α_i (filter FE)
    y += beta_dict[f"t_i{i}"] * future_t           # β_i · t
    y += beta_dict["sin1"] * future_sin1           # γ_1 sin
    y += beta_dict["cos1"] * future_cos1           # γ_2 cos
    y += beta_dict["H_m7"] * H_m7                  # δ_m H^{m,7}
    y += beta_dict["H_l7"] * H_l7                  # δ_l H^{l,7}
    y += beta_dict["A_m_f"] * A_m_f                # ρ_m \tilde{A}^m
    y += beta_dict["A_l_f"] * A_l_f                # ρ_l \tilde{A}^l
    y += beta_dict["has_m"] * has_m                # η_m 1^m
    y += beta_dict["has_l"] * has_l                # η_l 1^l

    return y, len(future_m), len(future_l)

def compute_life_days(i, y_sim):
    """用历史观测 + 未来模拟拼接后的序列做滚动 365d 均值，返回寿命（天）"""
    # 拼接历史 y 和未来 y_sim
    h_sub = hist[hist["i"] == i][["d","y"]].dropna().sort_values("d")
    h_days = h_sub["d"].values
    h_y = h_sub["y"].values
    # Future: future_days + y_sim
    all_dates = np.concatenate([h_days.astype("datetime64[D]"),
                                d_arr.astype("datetime64[D]")])
    all_y = np.concatenate([h_y, y_sim])
    order = np.argsort(all_dates)
    all_dates = all_dates[order]
    all_y = all_y[order]
    # 去重日期（若历史最后日与未来首日重叠）
    _, uniq_idx = np.unique(all_dates, return_index=True)
    uniq_idx = np.sort(uniq_idx)
    all_dates = all_dates[uniq_idx]
    all_y = all_y[uniq_idx]

    # 将序列转为 pandas 做滚动均值（兼容 NaN 容差）
    s = pd.Series(all_y, index=pd.to_datetime(all_dates))
    # 重采样为日频并前向填充观测区间的缺失（保留滚动自然处理 NaN）
    s = s.asfreq("D")
    ravg = s.rolling(window=365, min_periods=180).mean()

    # 只在未来段找首次 < 37
    fut_mask = ravg.index >= start_future
    ravg_fut = ravg[fut_mask]
    below = ravg_fut[ravg_fut < 37]
    if len(below) == 0:
        return np.inf, ravg_fut.iloc[-1] if len(ravg_fut) else np.nan
    L_date = below.index[0]
    L_days = (L_date - start_future).days
    return L_days, below.iloc[0]

def run_grid_search():
    # -------- 主循环 --------
    start_time = time.time()
    rows = []
    for i in range(1, 11):
        best = dict(EAC=np.inf)
        for T_M in T_M_grid:
            for T_L in T_L_grid:
                y_sim, _, _ = simulate_y(i, T_M, T_L)
                L_days, rav_at_L = compute_life_days(i, y_sim)
                if np.isinf(L_days):
                    L_years = 20.0  # 上限
                    retired = False
                else:
                    L_years = L_days / 365.25
                    retired = True
                # 成本
                n_M = L_years * 365.25 / T_M
                n_L = L_years * 365.25 / T_L if np.isfinite(T_L) else 0.0
                cost_total = C_BUY + n_M * C_M + n_L * C_L
                EAC = cost_total / L_years
                row = dict(
                    i=i, T_M=T_M, T_L=T_L if np.isfinite(T_L) else 99999,
                    T_L_label="inf" if np.isinf(T_L) else str(int(T_L)),
                    L_days=int(L_days) if np.isfinite(L_days) else 99999,
                    L_years=round(L_years, 3),
                    retired=retired,
                    n_M=round(n_M, 2),
                    n_L=round(n_L, 2),
                    cost_total=round(cost_total, 2),
                    EAC=round(EAC, 3),
                )
                rows.append(row)
                if EAC < best["EAC"]:
                    best = row.copy()
        elapsed = time.time() - start_time
        print(f"A{i:2d}: T_M*={best['T_M']}, T_L*={best['T_L_label']}, "
              f"L={best['L_years']:.2f}y, EAC*={best['EAC']:.2f} 万元/年  "
              f"(retired={best['retired']})  [{elapsed:.1f}s]")

    grid = pd.DataFrame(rows)
    grid.to_csv(ROOT / "Q3输出/tables/eac_grid_all.csv", index=False)
    print(f"\nSaved Q3输出/tables/eac_grid_all.csv  ({len(grid)} rows)")

    # -------- 每台最优 --------
    best_rows = []
    for i in range(1, 11):
        sub = grid[grid["i"] == i]
        best = sub.loc[sub["EAC"].idxmin()].to_dict()
        best_rows.append(best)
    best_df = pd.DataFrame(best_rows)
    best_df.to_csv(ROOT / "Q3输出/tables/optimal_rule_per_filter.csv", index=False)
    print(f"\nOptimal rule per filter:")
    print(best_df[["i","T_M","T_L_label","L_years","retired","n_M","n_L","cost_total","EAC"]].to_string(index=False))


if __name__ == "__main__":
    run_grid_search()


A 1: T_M*=80, T_L*=450, L=4.24y, EAC*=94.27 万元/年  (retired=True)  [2.0s]
A 2: T_M*=100, T_L*=540, L=11.57y, EAC*=45.00 万元/年  (retired=True)  [4.0s]
A 3: T_M*=90, T_L*=540, L=3.78y, EAC*=99.64 万元/年  (retired=True)  [6.0s]
A 4: T_M*=120, T_L*=inf, L=20.00y, EAC*=24.13 万元/年  (retired=False)  [8.1s]
A 5: T_M*=60, T_L*=720, L=1.75y, EAC*=195.56 万元/年  (retired=True)  [10.2s]
A 6: T_M*=120, T_L*=inf, L=20.00y, EAC*=24.13 万元/年  (retired=False)  [12.2s]
A 7: T_M*=100, T_L*=720, L=3.10y, EAC*=113.93 万元/年  (retired=True)  [14.2s]
A 8: T_M*=80, T_L*=300, L=2.09y, EAC*=172.11 万元/年  (retired=True)  [16.2s]
A 9: T_M*=90, T_L*=540, L=2.69y, EAC*=131.76 万元/年  (retired=True)  [18.2s]
A10: T_M*=50, T_L*=240, L=1.07y, EAC*=321.14 万元/年  (retired=True)  [20.3s]

Saved Q3输出/tables/eac_grid_all.csv  (810 rows)

Optimal rule per filter:
 i  T_M T_L_label  L_years  retired   n_M  n_L  cost_total     EAC
 1   80       450    4.235     True 19.34 3.44      399.27  94.268
 2  100       540   11.573     True 42.27 

In [25]:
"""
Q3.2  当前规律 vs 最优规律 对比
  - 当前规律：Q2 fixed_maintenance_rule.csv 的 T_M_use, T_L_use
  - 当前 L：Q2 life_prediction.csv 的 L_year（inf 截断到 20 年作为上限）
  - 最优规律：Q3 step1 的 optimal_rule_per_filter.csv

输出：
  tables/comparison_current_vs_optimal.csv
"""
import pandas as pd
import numpy as np
from pathlib import Path

PROJECT_ROOT = Path(r"d:\Users\32174\Desktop\2026_math_modeling_competition")
ROOT = PROJECT_ROOT  # 修正：指向项目根目录


In [26]:

rule_now = pd.read_csv(ROOT / "Q2输出/tables/fixed_maintenance_rule.csv")
life_now = pd.read_csv(ROOT / "Q2输出/tables/life_prediction.csv")
opt = pd.read_csv(ROOT / "Q3输出/tables/optimal_rule_per_filter.csv")

C_BUY = 300.0
C_M = 3.0
C_L = 12.0
L_CAP = 20.0  # 与 Q3 仿真地平线一致


In [27]:

rows = []
for i in range(1, 11):
    rn = rule_now[rule_now["i"] == i].iloc[0]
    ln = life_now[life_now["i"] == i].iloc[0]
    op = opt[opt["i"] == i].iloc[0]

    # 当前规律
    T_M_cur = rn["T_M_use"]
    T_L_cur = rn["T_L_use"] if not (isinstance(rn["T_L_use"], str) and rn["T_L_use"] == "inf") else np.inf
    try:
        T_L_cur = float(T_L_cur)
    except Exception:
        T_L_cur = np.inf

    L_cur_y_raw = ln["L_year"]
    if isinstance(L_cur_y_raw, str) and L_cur_y_raw == "inf":
        L_cur_y = L_CAP
        retired_cur = False
    elif np.isinf(L_cur_y_raw) or pd.isna(L_cur_y_raw):
        L_cur_y = L_CAP
        retired_cur = False
    else:
        L_cur_y = float(L_cur_y_raw)
        retired_cur = True

    n_M_cur = L_cur_y * 365.25 / T_M_cur
    n_L_cur = L_cur_y * 365.25 / T_L_cur if np.isfinite(T_L_cur) else 0.0
    cost_cur = C_BUY + n_M_cur * C_M + n_L_cur * C_L
    EAC_cur = cost_cur / L_cur_y

    # 最优
    T_M_opt = op["T_M"]
    T_L_opt_lbl = op["T_L_label"]
    T_L_opt = np.inf if T_L_opt_lbl == "inf" else float(T_L_opt_lbl)
    L_opt_y = op["L_years"]
    EAC_opt = op["EAC"]
    cost_opt = op["cost_total"]
    n_M_opt = op["n_M"]
    n_L_opt = op["n_L"]
    retired_opt = bool(op["retired"])

    # 节省
    save_abs = EAC_cur - EAC_opt
    save_pct = save_abs / EAC_cur * 100

    rows.append(dict(
        i=i,
        T_M_cur=T_M_cur,
        T_L_cur=("inf" if np.isinf(T_L_cur) else f"{T_L_cur:g}"),
        L_cur_years=round(L_cur_y, 2),
        retired_cur=retired_cur,
        n_M_cur=round(n_M_cur, 1),
        n_L_cur=round(n_L_cur, 1),
        cost_cur=round(cost_cur, 1),
        EAC_cur=round(EAC_cur, 2),
        T_M_opt=int(T_M_opt),
        T_L_opt=T_L_opt_lbl,
        L_opt_years=round(L_opt_y, 2),
        retired_opt=retired_opt,
        n_M_opt=round(n_M_opt, 1),
        n_L_opt=round(n_L_opt, 1),
        cost_opt=round(cost_opt, 1),
        EAC_opt=round(EAC_opt, 2),
        save_abs=round(save_abs, 2),
        save_pct=round(save_pct, 1),
    ))

cmp = pd.DataFrame(rows)
print("当前规律 vs 最优规律:")
cols_print = ["i","T_M_cur","T_L_cur","L_cur_years","EAC_cur",
              "T_M_opt","T_L_opt","L_opt_years","EAC_opt","save_abs","save_pct"]
print(cmp[cols_print].to_string(index=False))

cmp.to_csv(ROOT / "Q3输出/tables/comparison_current_vs_optimal.csv", index=False)
print("\nSaved Q3输出/tables/comparison_current_vs_optimal.csv")

# 汇总
total_cur = cmp["EAC_cur"].sum()
total_opt = cmp["EAC_opt"].sum()
total_save = total_cur - total_opt
total_save_pct = total_save / total_cur * 100
print(f"\n10 台合计年化成本：")
print(f"  当前规律: {total_cur:.2f} 万元/年")
print(f"  最优规律: {total_opt:.2f} 万元/年")
print(f"  节省:     {total_save:.2f} 万元/年  ({total_save_pct:.1f}%)")

# A4/A6 警示
print("\n⚠ 警示：A4 / A6 在 Q2 模型下历史斜率为正 (β>0)，")
print("   其外推 y 会持续上涨，寿命预测不退役，EAC 为保守下限估计。")
print("   实际工程中应按 β<0 的其它台类比处理，或使用更短历史窗口重新估 β。")


当前规律 vs 最优规律:
 i  T_M_cur T_L_cur  L_cur_years  EAC_cur  T_M_opt  T_L_opt  L_opt_years  EAC_opt  save_abs  save_pct
 1     62.0     347         4.46    97.53       80    450.0         4.24    94.27      3.26       3.3
 2     56.5    57.5        20.00   110.62      100    540.0        11.57    45.00     65.62      59.3
 3     55.5     138         4.28   121.65       90    540.0         3.78    99.64     22.02      18.1
 4     57.0     inf        20.00    34.22      120      inf        20.00    24.13     10.09      29.5
 5     60.0   262.5         1.87   195.39       60    720.0         1.75   195.56     -0.17      -0.1
 6     71.0     105        20.00    72.18      120      inf        20.00    24.13     48.04      66.6
 7     62.0     298         3.49   118.46      100    720.0         3.10   113.93      4.53       3.8
 8     54.5     inf         1.80   186.89       80    300.0         2.09   172.11     14.78       7.9
 9     62.5     321         2.97   132.08       90    540.0         

In [31]:
"""
Q3.2b  用 12 年地平线重新评估"当前规律"的真实寿命和 EAC
  (Q2 只跑了 10 年，对"不退役"的 5 台给出 L=inf → 截断 10y 导致 EAC 低估)
  本步把 10 台都在 12 年地平线下重新模拟，直接对比 Q3 最优方案
"""
import sys
import pandas as pd
import numpy as np
from pathlib import Path

# 把当前 code 目录加入 sys.path 以便 import step1_grid_search_eac
sys.path.insert(0, str(Path(__file__).resolve().parent))
from step1_grid_search_eac import simulate_y, compute_life_days  # 复用

PROJECT_ROOT = Path(r"d:\Users\32174\Desktop\2026_math_modeling_competition")
ROOT = PROJECT_ROOT  # 修正：指向项目根目录

rule = pd.read_csv(ROOT / "Q2输出/tables/fixed_maintenance_rule.csv")

C_BUY = 300.0
C_M = 3.0
C_L = 12.0
L_CAP = 12.0  # 12 年地平线 (覆盖正常退役最长 A2 11.57y, 不再摊薄异常台 cost_buy)

rows = []
for _, r in rule.iterrows():
    i = int(r["i"])
    T_M = float(r["T_M_use"])
    T_L_raw = r["T_L_use"]
    try:
        T_L = float(T_L_raw)
    except Exception:
        T_L = np.inf
    if str(T_L_raw) == "inf":
        T_L = np.inf

    y_sim, _, _ = simulate_y(i, T_M, T_L)
    L_days, _ = compute_life_days(i, y_sim)
    if np.isinf(L_days):
        L_years = L_CAP
        retired = False
    else:
        L_years = L_days / 365.25
        retired = True

    n_M = L_years * 365.25 / T_M
    n_L = L_years * 365.25 / T_L if np.isfinite(T_L) else 0.0
    cost_total = C_BUY + n_M * C_M + n_L * C_L
    EAC = cost_total / L_years

    rows.append(dict(
        i=i, T_M=T_M,
        T_L=("inf" if np.isinf(T_L) else f"{T_L:g}"),
        L_years_12h=round(L_years, 3),
        retired_12h=retired,
        n_M=round(n_M, 2),
        n_L=round(n_L, 2),
        cost_total=round(cost_total, 2),
        EAC=round(EAC, 2),
    ))

df = pd.DataFrame(rows)
print("当前规律在 12 年地平线下的重新评估:")
print(df.to_string(index=False))
df.to_csv(ROOT / "Q3输出/tables/current_rule_12y.csv", index=False)
print("\nSaved Q3输出/tables/current_rule_12y.csv")


NameError: name '__file__' is not defined